In [1]:
from lib.db.crud.publication import get_or_create_publication
from lib.db.crud.authors.get_or_create import get_or_create_author
from lib.helpers.similarity.main import check_similarity
from datetime import date

In [1]:
from bs4 import BeautifulSoup


def read_cv(path_cv):
    with open(path_cv, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    return soup

path_html = 'data/curriculos/2747150211073176/cv.html'
soup = read_cv(path_html)

In [2]:
from sqlalchemy.orm import sessionmaker
from lib.db.database import engine

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

# Create Profile

In [3]:
from lib.parser.lattes.profile import save_profile


lattes_id = '2747150211073176'

In [ ]:
save_profile(soup, lattes_id)

In [ ]:
from lib.db.crud.authors.profile import get_or_create_profile

author_db = get_or_create_profile(session, lattes_id)
author_db

In [8]:
author_db.id

3

# Atualiza Curiculos

In [1]:
from lib.db.make_session import local_session
from lib.updates.check_cv import check_cv_update

In [ ]:
session = local_session()
lattes_id = '2747150211073176'  
html, update = check_cv_update(session, lattes_id)

# Artigos sem doi

In [9]:

from lib.db.crud.authors.vinculate_publication import authors_to_publication
from lib.db.crud.container import get_or_create_container
from lib.parser.crossref.author_crossref import parser_contributor
from lib.parser.lattes.container import parser_container_lattes
from lib.parser.lattes.contributor import parser_contributor_lattes
from lib.parser.lattes.publi_sem_doi import parser_publi_sem_doi

In [3]:
from lib.parser.lattes.artigos_completos import get_artigos_completos, slipt_artigos
list_artigos = get_artigos_completos(soup)
c_doi, s_doi = slipt_artigos(list_artigos)

In [8]:
import json


with open("data/curriculos/2747150211073176/article_sem_doi.jsonl", "w", encoding="utf-8") as f:
    for article in s_doi:
        del article['raw_artigo']
        json.dump(article, f, ensure_ascii=False)
        f.write("\n")
        
    

In [ ]:
for article in s_doi:
    publication = parser_publi_sem_doi(article)
    container = parser_container_lattes(article)
    
    publication_db = get_or_create_publication(session, publication)
    container_db = get_or_create_container(session, container)
    publication_db.container = container_db
    
    authors = article["autores"]
    for author in authors:
        parsed_author = parser_contributor_lattes(author)
        contributor = parser_contributor(author, parsed_author)
        author_db, created = get_or_create_author(session, parsed_author)
        link = authors_to_publication(session, publication_db, author_db, contributor )
        print("FEITO: ", link)
        
    session.add(publication_db)
    session.commit() 
    
    


# Artigos com doi

In [ ]:
from lib.crossref.get import get_article_crossref


error = get_article_crossref(c_doi, lattes_id)

In [8]:
import json


articles = []
with open('data/curriculos/2747150211073176/article_crossref.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        artigo = json.loads(line)
        articles.append(artigo)
len(articles)

258

In [ ]:
from lib.db.crud.articles import injest_article


injest_article(articles)

In [ ]:
error = []
with open("data/artigos/val.jsonl", "w", encoding="utf-8") as f:
    
    for i in c_doi:
        doi = i['doi'][0]
        url = f"https://api.crossref.org/v1/works/{doi}"
        r = httpx.get(url)
        print(r.status_code)
        if r.status_code == 200:
            item = r.json()['message']
            json.dump(item, f)
            f.write("\n")
        else:
            print(f"Error fetching data for DOI: {doi}, status code: {r.status_code}")
            error.append(i)

# Livros

In [4]:
import json

In [5]:
livros = []
with open('data/curriculos/2747150211073176/livros.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        livro = json.loads(line)
        livros.append(livro)
        
len(livros)
    

24

In [6]:
from lib.parser.crossref.date import parse_date


livro = livros[0]
publication = {
        'publication_type': 'book',
        'title': livro.get('titulo'),
        'date_published': date(int(livro.get('ano')), 1, 1),
        'doi': livro.get('doi'),
        'publisher': livro.get('editora'),
        'volume_number': livro.get('volume'),
        'number_of_pages': livro.get('paginas'),
        'edition': livro.get('edicao'),
        'source': 'lattes'        
    }
publication

{'publication_type': 'book',
 'title': 'Micropláticos: um problema complexo e urgente',
 'date_published': datetime.date(2025, 1, 1),
 'doi': None,
 'publisher': 'Editora ABC',
 'volume_number': 1,
 'number_of_pages': 26,
 'edition': 1,
 'source': 'lattes'}

In [ ]:
publication_db = get_or_create_publication(session, publication)


In [12]:
autores = livro['autores']
autor = autores[0]
autor

{'given_name': 'A. L.',
 'family_name': 'Val',
 'full_name': 'A. L. Val',
 'normalized_full_name': 'a. l. val',
 'canonical_source': 'lattes'}

In [14]:
author_db, created = get_or_create_author(session, autor)

2026-04-26 22:24:58,471 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors
2026-04-26 22:24:58,475 INFO sqlalchemy.engine.Engine [generated in 0.00396s] {}


In [15]:
author_db.id

3

In [ ]:
for livro in livros:
    number_of_pages = livro.get('paginas') 
    if number_of_pages == '':
        number_of_pages = None
    publication = {
        'publication_type': 'book',
        'title': livro.get('titulo'),
        'date_published': date(int(livro.get('ano')), 1, 1),
        'doi': livro.get('doi'),
        'publisher': livro.get('editora'),
        'volume_number': livro.get('volume'),
        'number_of_pages': number_of_pages,
        'edition': livro.get('edicao'),
        'source': 'lattes'        
    }
    
    publication_db = get_or_create_publication(session, publication)
    authors = livro["autores"]
    for author in authors:
        author_db, created = get_or_create_author(session, author)
        contributor = {'role': 'author'}
        link = authors_to_publication(session, publication_db, author_db, contributor )
        print(author)

2026-04-26 22:49:42,356 INFO sqlalchemy.engine.Engine SELECT publications.id AS publications_id, publications.publication_type AS publications_publication_type, publications.title AS publications_title, publications.subtitle AS publications_subtitle, publications.alternative_title AS publications_alternative_title, publications.abstract AS publications_abstract, publications.date_published AS publications_date_published, publications.language AS publications_language, publications.subject AS publications_subject, publications.doi AS publications_doi, publications.isbn AS publications_isbn, publications.identifier AS publications_identifier, publications.publisher AS publications_publisher, publications.url AS publications_url, publications.license AS publications_license, publications.conditions_of_access AS publications_conditions_of_access, publications.is_accessible_for_free AS publications_is_accessible_for_free, publications.is_part_of_id AS publications_is_part_of_id, publication

DataError: (pymysql.err.DataError) (1366, "Incorrect integer value: '' for column `orbis_db`.`publications`.`number_of_pages` at row 1")
[SQL: INSERT INTO publications (publication_type, title, subtitle, alternative_title, abstract, date_published, language, subject, doi, isbn, identifier, publisher, url, license, conditions_of_access, is_accessible_for_free, is_part_of_id, page_start, page_end, volume_number, issue_number, edition, number_of_pages, in_support_of, source_organization, source, needs_review) VALUES (%(publication_type)s, %(title)s, %(subtitle)s, %(alternative_title)s, %(abstract)s, %(date_published)s, %(language)s, %(subject)s, %(doi)s, %(isbn)s, %(identifier)s, %(publisher)s, %(url)s, %(license)s, %(conditions_of_access)s, %(is_accessible_for_free)s, %(is_part_of_id)s, %(page_start)s, %(page_end)s, %(volume_number)s, %(issue_number)s, %(edition)s, %(number_of_pages)s, %(in_support_of)s, %(source_organization)s, %(source)s, %(needs_review)s) RETURNING publications.id, pu

In [11]:
livro

{'doi': 'http://dx.doi.org/10.55161/RWSX6527',
 'autores': [{'given_name': 'C. A.',
   'family_name': 'Nobre',
   'full_name': 'C. A. Nobre',
   'normalized_full_name': 'c. a. nobre',
   'canonical_source': 'lattes'},
  {'given_name': 'A.',
   'family_name': 'Encaladi',
   'full_name': 'A. Encaladi',
   'normalized_full_name': 'a. encaladi',
   'canonical_source': 'lattes'},
  {'given_name': 'E.',
   'family_name': 'Anderson',
   'full_name': 'E. Anderson',
   'normalized_full_name': 'e. anderson',
   'canonical_source': 'lattes'},
  {'given_name': 'F. A.',
   'family_name': 'Roca',
   'full_name': 'F. A. Roca',
   'normalized_full_name': 'f. a. roca',
   'canonical_source': 'lattes'},
  {'given_name': 'M.',
   'family_name': 'Bustammante',
   'full_name': 'M. Bustammante',
   'normalized_full_name': 'm. bustammante',
   'canonical_source': 'lattes'},
  {'given_name': 'C.',
   'family_name': 'Mena',
   'full_name': 'C. Mena',
   'normalized_full_name': 'c. mena',
   'canonical_source':